In [15]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 벡터 DB : Chroma vs. Pinecone
- Chroma : 인메모리 vector DB, 로컬 vector DB, collection단위로 저장
- Pinecone : 클라우드 vector DB, index단위로 저장
(http://www.pinecone.io)에서 api key 받아 .env추가(PINECONE_API_KEY등록)

# 0. 패키지 설치

In [16]:
%pip install -q pinecone langchain-pinecone # --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


# 1. knowledge base 구성을 위한 데이터 생성

In [17]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법(법률)(제21221호)(20260701).docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 7.08 s
Wall time: 7.1 s


180

In [18]:
# 임베딩 객체
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')

In [ ]:
# # 임베딩 객체
# from dotenv import load_dotenv
# from langchain_upstage import UpstageEmbeddings
# load_dotenv()
# embedding = UpstageEmbeddings(model = 'solar-embedding-1-large-passage')

In [19]:
len(embedding.embed_query("샘플"))

3072

In [ ]:
# 모델마다 코사인유사도를 쓰는지 document
# https://console.upstage.ai/docs/capabilities/embed
# https://developers.openai.com/api/docs/guides/embeddings

In [20]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
import os
pc = Pinecone(
    # api_key=os.getenv('PINECONE_API_KEY')
)
# 데이터를 처음 저장할 때
index_name = 'tax-index-openai'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드된 벡터 DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name
)

CPU times: total: 0 ns
Wall time: 2.09 ms


# 2. 답변 생성을 위한 Retrieval

In [21]:
query = "연봉 5천만원인 직장인은 소득세를 얼마낼까요?"
retrieved_docs = database.similarity_search(query, k=3) # 기본k값은 4

In [22]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n','\n') for doc in retrieved_docs])
#print(retrieved_doc)

# 3. 답변생성

In [ ]:
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model='gpt-4.1-nano')

In [23]:
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model = 'solar-pro2')

In [24]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가입니다. [context]를 참고해서 사용자 질문에 답하세요. [context]는 다음과 같아요
{retrieved_doc}
- 질문 : {query}"""

In [25]:
ai_message = llm.invoke(prompt)
print(ai_message.content)

2023년 한국 소득세법을 기준으로 연봉 5,000만원인 직장인의 소득세를 계산해드리겠습니다. 단, 추가 공제(의료비, 교육비 등)는 고려하지 않은 기본 계산입니다.

### 1. **근로소득금액 확인**
   - 연봉 5,000만원 = **근로소득금액 5,000만원**

### 2. **근로소득공제 적용** (소득세법 제50조)
   - 5,000만원 × 15% - 54만원 = **646만원 공제**  
   → **공제 후 금액: 4,354만원** (5,000 - 646)

### 3. **종합소득과세표준 계산** (기본공제만 적용)
   - **기본공제**: 본인 150만원 + 부양가족(배우자/직계존비속) 150만원 × 인원 수  
     (예시로 부양가족 1인 추가 시 → 300만원 공제)  
   - **계산**: 4,354만원 - 300만원 = **4,054만원** (과세표준)

### 4. **산출세액 계산** (2023년 세율 적용)
   - 1,200만원 × 6% = 72만원  
   - (4,054만원 - 1,200만원) × 15% = 432.3만원  
   → **산출세액: 504.3만원** (72 + 432.3)

### 5. **근로소득세액공제** (소득세법 제55조)
   - 504.3만원 × 55% = 277.365만원 → **최대 74만원 한도 적용**  
   → **공제 후 세액: 430.3만원** (504.3 - 74)

### 6. **월별 분할 납부** (연말정산 시)
   - 430.3만원을 12개월로 분할 납부  
     (월 약 **35.86만원**)

### ✨ **최종 납부세액**: **약 430만원**  
(※ 부양가족 없을 경우 516만원, 추가 공제 시 감소 가능)

#### 📌 참고사항
- **4대 보험** (국민연금, 건강보험 등)은 별도로 공제되며, 이 계산에서는 제외되었습니다.  
- 정확한 금액은 실제 연말정산 시 공제항목(의료비, 신용카드 등)에 따라 달라질 수 있습니다.  
- 2024년 세율

# 4. Langchain

In [26]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 1. 임베딩 & LLM 초기화
load_dotenv()
embedding = OpenAIEmbeddings(model = "text-embedding-3-large")
llm = ChatOpenAI(model='gpt-4.1-mini')
# 2. vector DB load
pc = Pinecone()
vectorStore = Chroma(
    embedding_function=embedding,
    persist_directory='./chroma',
    collection_name='tax-collection'
)
# 3. Retriever 생성
# (1) search_type='similarity' : 가장 유사한 top-k개 추출
retriever = vectorStore.as_retriever(search_type='similarity', 
                                     search_kwargs={'k':3}) # 유사도가 높은 top-3추출
# # (2) search_type='mmr' : 후보 10개 중 서로 덜 겹치는 3개 추출
# retriever = vectorStore.as_retriever(search_type='mmr', 
#                                      search_kwargs={'k':3,
#                                                    'fetch_k':10,
#                                                    'lambda_mult':0.5}) # 1에 가까울수록 관련성 깊은 계산
# # (3) search_type='similarity_score_threshold' : 
# retriever = vectorStore.as_retriever(search_type='similarity_score_threshold', 
#                                      search_kwargs={'k':3, 'score_threshold':0.4}) 
# 4. 프롬프트템플릿
template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 모른다고 
답하세요. 최대 3문장으로 간결하게 답변하세요
질문 : {{query}}
문맥 : {{context}}
답변 :"""
prompt = ChatPromptTemplate.from_template(template)
# 5. 검색된 문서를 텍스트로 변환하는 함수
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [27]:
# 6. RAG 체인 구성
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼마인가요?')

'연봉 5천만원인 직장인의 소득세는 과세표준에서 근로소득공제 등을 적용한 후, 해당 과세표준 구간에 따른 누진세율을 적용하여 계산합니다. 2024년 기준 대략적으로 근로소득공제 후 과세표준 약 4천만원 수준에서 세액은 약 300만~400만원대가 될 수 있으나, 정확한 금액은 개인별 공제사항과 세액공제 여부에 따라 달라집니다. 따라서 구체적 계산을 위해서는 연말정산 자료나 세무 상담이 필요합니다.'